# LoRA fine-tuning Qwen2.5-1.5B for named entity recognition

Fine-tunes a small instruction model with QLoRA (4-bit base + LoRA adapters) to extract named entities as JSON from news text, using the Stanford `en-worldwide-newswire` dataset. Runs on a free Colab T4.

See the README for results and discussion.

## 1. Get the data

In [ ]:
!git clone https://github.com/stanfordnlp/en-worldwide-newswire

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import glob, os
files = sorted(glob.glob('en-worldwide-newswire/processed_annotated/**/*.tsv', recursive=True))
print(len(files), 'files')
print(files[:3])
with open(files[0], encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i > 25: break

1075 files
['en-worldwide-newswire/processed_annotated/af_afrol_1.txt.tsv', 'en-worldwide-newswire/processed_annotated/af_afrol_10.txt.tsv', 'en-worldwide-newswire/processed_annotated/af_afrol_11.txt.tsv']
'http://www.afrol.com/articles/37145\tO\n'
'\n'
'afrol\tB-Organization\n'
'News\tI-Organization\n'
',\tO\n'
'27\tB-Date\n'
'January\tI-Date\n'
'-\tO\n'
'David\tB-Person\n'
'Kato\tI-Person\n'
...


## 2. Convert token-level BIO tags into text → entity JSON examples

Sentences are grouped into chunks of up to ~120 tokens. The split is **by article** (80/10/10), so no article's text appears in both train and test.

In [ ]:
import json, random
from collections import Counter

def read_sentences(path):
    sents, cur = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if cur: sents.append(cur); cur = []
                continue
            parts = line.split('\t')
            if len(parts) < 2: continue
            word, tag = parts[0], parts[-1]
            if word.startswith('http'): continue
            cur.append((word, tag))
            if word in {'.', '?', '!'} and tag == 'O':
                sents.append(cur); cur = []
    if cur: sents.append(cur)
    return sents

def extract_entities(tokens):
    ents, span, etype = [], [], None
    for word, tag in tokens:
        if tag == 'O' or '-' not in tag:
            if span: ents.append((etype, ' '.join(span)))
            span, etype = [], None
            continue
        pos, typ = tag.split('-', 1)
        if pos in ('B', 'S') or typ != etype:
            if span: ents.append((etype, ' '.join(span)))
            span, etype = [word], typ
        else:
            span.append(word)
        if pos in ('E', 'S'):
            ents.append((etype, ' '.join(span)))
            span, etype = [], None
    if span: ents.append((etype, ' '.join(span)))
    return ents

def make_examples(path, max_tokens=120):
    doc = os.path.basename(path)
    country = doc.split('_')[0]
    examples, chunk = [], []
    def flush(chunk):
        if not chunk: return
        ents = {}
        for t, e in extract_entities(chunk):
            ents.setdefault(t, [])
            if e not in ents[t]: ents[t].append(e)
        examples.append({'doc': doc, 'country': country,
                         'text': ' '.join(w for w, _ in chunk),
                         'entities': ents})
    for sent in read_sentences(path):
        if chunk and len(chunk) + len(sent) > max_tokens:
            flush(chunk); chunk = []
        chunk.extend(sent)
    flush(chunk)
    return examples

all_docs = [make_examples(p) for p in files]
random.seed(42)
random.shuffle(all_docs)
n = len(all_docs)
splits = {'train': all_docs[:int(.8*n)],
          'val':   all_docs[int(.8*n):int(.9*n)],
          'test':  all_docs[int(.9*n):]}

out = '/content/drive/MyDrive/ner-lora'
os.makedirs(out, exist_ok=True)
for name, docs in splits.items():
    rows = [ex for d in docs for ex in d]
    with open(f'{out}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'{name}: {len(docs)} articles, {len(rows)} examples')

labels = Counter(t for d in all_docs for ex in d for t, v in ex['entities'].items() for _ in v)
print(labels)
print(json.dumps(all_docs[0][0], indent=2, ensure_ascii=False)[:800])

train: 860 articles, 5444 examples
val: 107 articles, 704 examples
test: 108 articles, 668 examples
Counter({'Location': 11527, 'Person': 9587, 'Organization': 8256, 'NORP': 4515, 'Date': 4217, 'Misc': 2944, 'Money': 952, 'Facility': 567, 'Product': 260})
{
  "doc": "israel_haaretz_8.txt.tsv",
  "country": "israel",
  "text": "The government is paying millions of shekels in compensation to mixed Jewish - Arab cities ...",
  "entities": {
    "Money": ["millions of shekels"],
    "NORP": ["Jewish", "Arab"],
    "Misc": ["Operation Guardian of the Walls"],
    "Date": ["May 2021"],
    "Location": ["Israel"]
  }
}


## 3. Load the base model (4-bit) and build prompting + scoring

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import torch, json, random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', dtype=torch.float16)

In [ ]:
from collections import defaultdict
out = '/content/drive/MyDrive/ner-lora'
def load(name): return [json.loads(l) for l in open(f'{out}/{name}.jsonl', encoding='utf-8')]
train, val = load('train'), load('val')

TYPES = ['Person','Location','Organization','Facility','Date','Money','NORP','Product','Misc']
SYSTEM = ("Extract named entities from the text. Return only JSON mapping each type to a list "
          "of exact strings from the text. Types: Person, Location, Organization, Facility "
          "(buildings, airports, roads), Date, Money, NORP (nationalities, religious or political "
          "groups), Product, Misc (events, laws, other names). Omit types with no entities.")

def build_messages(text, shots=0):
    msgs = [{'role': 'system', 'content': SYSTEM}]
    for ex in train[:shots]:
        msgs += [{'role': 'user', 'content': ex['text']},
                 {'role': 'assistant', 'content': json.dumps(ex['entities'], ensure_ascii=False)}]
    msgs.append({'role': 'user', 'content': text})
    return msgs

def parse(output):
    s, e = output.find('{'), output.rfind('}')
    if s == -1 or e == -1: return None
    try:
        d = json.loads(output[s:e+1])
        return {k: [str(x) for x in v] for k, v in d.items() if isinstance(v, list)}
    except Exception:
        return None

@torch.no_grad()
def predict(texts, shots=0, batch_size=8):
    tok.padding_side = 'left'
    outs = []
    for i in range(0, len(texts), batch_size):
        prompts = [tok.apply_chat_template(build_messages(t, shots), tokenize=False,
                                           add_generation_prompt=True) for t in texts[i:i+batch_size]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=300, do_sample=False)
        outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return outs

norm = lambda s: ''.join(s.lower().split())

def score(examples, outputs):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    valid = 0
    for ex, o in zip(examples, outputs):
        pred = parse(o)
        valid += pred is not None
        pred = pred or {}
        for t in TYPES:
            g = {norm(x) for x in ex['entities'].get(t, [])}
            p = {norm(x) for x in pred.get(t, [])}
            tp[t] += len(g & p); fp[t] += len(p - g); fn[t] += len(g - p)
    def prf(a, b, c):
        p = a/(a+b) if a+b else 0; r = a/(a+c) if a+c else 0
        return p, r, (2*p*r/(p+r) if p+r else 0)
    print(f'Valid JSON: {valid/len(examples):.1%}')
    for t in TYPES:
        p, r, f = prf(tp[t], fp[t], fn[t]); print(f'{t:13s} P={p:.2f} R={r:.2f} F1={f:.2f}')
    p, r, f = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    print(f'{"OVERALL":13s} P={p:.2f} R={r:.2f} F1={f:.2f}')

## 4. Baselines: zero-shot and 3-shot prompting (200 validation examples)

In [ ]:
sample = random.Random(0).sample(val, 200)
texts = [ex['text'] for ex in sample]

print('=== Zero-shot ===')
zero = predict(texts, shots=0); score(sample, zero)

print('\n=== 3-shot ===')
few = predict(texts, shots=3); score(sample, few)

=== Zero-shot ===
Valid JSON: 42.5%
Person        P=0.63 R=0.26 F1=0.36
Location      P=0.65 R=0.17 F1=0.27
Organization  P=0.16 R=0.04 F1=0.06
Facility      P=0.00 R=0.00 F1=0.00
Date          P=0.40 R=0.07 F1=0.12
Money         P=0.00 R=0.00 F1=0.00
NORP          P=0.00 R=0.00 F1=0.00
Product       P=0.00 R=0.00 F1=0.00
Misc          P=0.03 R=0.07 F1=0.04
OVERALL       P=0.34 R=0.12 F1=0.18

=== 3-shot ===
Valid JSON: 98.5%
Person        P=0.52 R=0.60 F1=0.56
Location      P=0.59 R=0.14 F1=0.22
Organization  P=0.50 R=0.01 F1=0.02
Facility      P=0.24 R=0.36 F1=0.29
Date          P=0.53 R=0.08 F1=0.14
Money         P=0.80 R=0.19 F1=0.31
NORP          P=0.04 R=0.01 F1=0.01
Product       P=0.00 R=0.00 F1=0.00
Misc          P=0.02 R=0.05 F1=0.03
OVERALL       P=0.31 R=0.19 F1=0.24


In [ ]:
for ex, o in list(zip(sample, few))[:3]:
    print('TEXT:', ex['text'][:300])
    print('GOLD:', ex['entities'])
    print('PRED:', o[:500])
    print('---')

TEXT: The Popular Union ( UP ) opposition party last week had announced protest marches in Malabo and Bata for yesterday , 23 March , at ten o'clock in the morning . ...
GOLD: {'Organization': ['Popular Union', 'UP'], 'Location': ['Malabo', 'Bata'], 'Date': ['23 March', 'Tuesday'], 'NORP': ['Equatoguineans', 'Equatoguinean'], 'Person': ['Clemente Engonga Nguema']}
PRED: {"Person": ["UP", "Equatoguineans", "Malabo", "Bata", "Clemente Engonga Nguema"], "Misc": ["Protest marches", "Marches", "Manifestations", "Reintroduction of basis liberties", "Improved social conditions"]}
---
TEXT: The affidavit was filed by the SIT 's Investigating Officer B C Solanki before the City Civil and Sessions Court in Ahmedabad on Friday . ...
GOLD: {'Organization': ['SIT', 'City Civil and Sessions Court'], 'Person': ['B C Solanki', 'Setalvad'], 'Location': ['Ahmedabad', 'State of Gujarat'], 'Date': ['Friday']}
PRED: {"Person": ["Setalvad", "Solanki", "B C Solanki"], "Product": ["affidavit"], "Misc": ["poli

## 5. QLoRA fine-tune

LoRA rank 16 on all attention and MLP projections, 2 epochs, effective batch size 16. Trainable LoRA weights are cast to fp32 to avoid fp16 gradient-scaling errors on the T4.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def to_pc(ex):
    return {'prompt': build_messages(ex['text'], shots=0),
            'completion': [{'role': 'assistant',
                            'content': json.dumps(ex['entities'], ensure_ascii=False)}]}

train_ds = Dataset.from_list([to_pc(e) for e in train])
val_ds = Dataset.from_list([to_pc(e) for e in random.Random(1).sample(val, 200)])

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'])

cfg = SFTConfig(output_dir=f'{out}/run1', num_train_epochs=2,
                per_device_train_batch_size=4, gradient_accumulation_steps=4,
                learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
                logging_steps=20, eval_strategy='steps', eval_steps=100,
                save_steps=100, save_total_limit=2, fp16=True, max_length=768,
                gradient_checkpointing=False, report_to='none')

tok.padding_side = 'right'
model.config.use_cache = False
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
                     eval_dataset=val_ds, peft_config=lora, processing_class=tok)

for name, p in trainer.model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainer.train()
trainer.save_model(f'{out}/run1/final')

## 6. Evaluate the fine-tuned model (same 200 validation examples, zero-shot prompt)

In [ ]:
sample = random.Random(0).sample(val, 200)
texts = [ex['text'] for ex in sample]
model = trainer.model
model.eval(); model.config.use_cache = True
print('=== Fine-tuned (zero-shot) ===')
ft = predict(texts, shots=0); score(sample, ft)

=== Fine-tuned (zero-shot) ===
Valid JSON: 99.5%
Person        P=0.94 R=0.93 F1=0.94
Location      P=0.91 R=0.87 F1=0.89
Organization  P=0.76 R=0.75 F1=0.75
Facility      P=0.33 R=0.29 F1=0.31
Date          P=0.95 R=0.96 F1=0.95
Money         P=0.95 R=0.95 F1=0.95
NORP          P=0.92 R=0.86 F1=0.89
Product       P=0.00 R=0.00 F1=0.00
Misc          P=0.57 R=0.44 F1=0.50
OVERALL       P=0.87 R=0.84 F1=0.86
